### Config

In [1]:
import os
from google.cloud import aiplatform
from dotenv import load_dotenv
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from google.cloud import storage
import vertexai
import numpy as np
import pickle
import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import os
from google.cloud.aiplatform.prediction import LocalModel
import joblib
import numpy as np
import pickle

from google.cloud.aiplatform.prediction.predictor import Predictor
from google.cloud.aiplatform.utils import prediction_utils

from sklearn.datasets import load_iris
import logging
import pickle

import joblib
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from google.cloud import aiplatform
logging.basicConfig(level=logging.INFO)

load_dotenv() 

PROJECT_ID = os.environ["PROJECT_ID"]
LOCATION = os.environ["LOCATION"]

aiplatform.init(project=PROJECT_ID, location=REGION)



2026-03-04 11:11:52.921545: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### Model

In [2]:
class MySimpleScaler(object):
    def __init__(self):
        self._means = None
        self._stds = None

    def preprocess(self, data):
        if self._means is None:  # during training only
            self._means = np.mean(data, axis=0)

        if self._stds is None:  # during training only
            self._stds = np.std(data, axis=0)
            if not self._stds.all():
                raise ValueError("At least one column has standard deviation of 0.")

        return (data - self._means) / self._stds

In [3]:
iris = load_iris()
scaler = MySimpleScaler()

In [4]:
X = scaler.preprocess(iris.data)
y = iris.target

In [5]:
model = RandomForestClassifier()
model.fit(X, y)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [6]:
X_test = [
    [6.7, 3.1, 4.7, 1.5],
    [4.6, 3.1, 1.5, 0.2]
]
inputs = np.asarray(X_test)
preprocessed_inputs = scaler.preprocess(inputs)

In [7]:
y_test = model.predict(preprocessed_inputs)
y_test

array([1, 0])

In [8]:
class_names = iris.target_names

In [9]:
predictions = {"predictions": [class_names[class_num] for class_num in y_test]}

In [10]:
predictions

{'predictions': [np.str_('versicolor'), np.str_('setosa')]}

### Do The Same Like Local Model Create Model and Artifact Folder

In [11]:
RANDOM_FOREST_CLASSIFIER_DIR = "random_forest_classifier"

In [12]:
RANDOM_FOREST_CLASSIFIER_ARTIFACT_DIR = "random_forest_classifier_artifacts"

### Build Docker

In [13]:
from random_forest_classifier.predictor import CprPredictor

In [14]:
REGION = "us-central1"
REPOSITORY = "random-forest-classifier-repo"
IMAGE = "random-forest-classifier-app"

In [15]:
f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPOSITORY}/{IMAGE}"

'us-central1-docker.pkg.dev/ridwan-faturrahman/random-forest-classifier-repo/random-forest-classifier-app'

In [16]:
DOCKER_IMAGE_NAME = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPOSITORY}/{IMAGE}"

In [17]:
local_model = LocalModel.build_cpr_model(
    RANDOM_FOREST_CLASSIFIER_DIR,
    DOCKER_IMAGE_NAME,
    predictor=CprPredictor,
    requirements_path=os.path.join(RANDOM_FOREST_CLASSIFIER_DIR, "requirements.txt"),
)

INFO:google.cloud.aiplatform.docker_utils.build:Running command: docker build -t us-central1-docker.pkg.dev/ridwan-faturrahman/random-forest-classifier-repo/random-forest-classifier-app --rm -f- random_forest_classifier
/home/ridwanfatur/miniconda3/envs/py3_11_13/lib/python3.11/subprocess.py:1010: RuntimeWarning: line buffering (buffering=1) isn't supported in binary mode, the default buffer size will be used
  self.stdin = io.open(p2cwrite, 'wb', bufsize)
/home/ridwanfatur/miniconda3/envs/py3_11_13/lib/python3.11/subprocess.py:1016: RuntimeWarning: line buffering (buffering=1) isn't supported in binary mode, the default buffer size will be used
  self.stdout = io.open(c2pread, 'rb', bufsize)
INFO:google.cloud.aiplatform.docker_utils.local_util:#0 building with "default" instance using docker driver

INFO:google.cloud.aiplatform.docker_utils.local_util:

INFO:google.cloud.aiplatform.docker_utils.local_util:#1 [internal] load build definition from Dockerfile

INFO:google.cloud.aiplatfor

### Push Image

In [20]:
!gcloud artifacts repositories create {REPOSITORY} \
    --repository-format=docker \
    --location=$REGION

Create request issued for: [random-forest-classifier-repo]
Waiting for operation [projects/ridwan-faturrahman/locations/us-central1/operat
ions/9938bd66-5b2f-4d25-8d44-e26e85ed15b0] to complete...done.                 
Created repository [random-forest-classifier-repo].


In [18]:
deployed_local_model = LocalModel(
    serving_container_image_uri=DOCKER_IMAGE_NAME,
    serving_container_predict_route="/predict",
    serving_container_health_route="/health"
)

In [21]:
deployed_local_model.push_image()

/home/ridwanfatur/miniconda3/envs/py3_11_13/lib/python3.11/subprocess.py:1010: RuntimeWarning: line buffering (buffering=1) isn't supported in binary mode, the default buffer size will be used
  self.stdin = io.open(p2cwrite, 'wb', bufsize)
/home/ridwanfatur/miniconda3/envs/py3_11_13/lib/python3.11/subprocess.py:1016: RuntimeWarning: line buffering (buffering=1) isn't supported in binary mode, the default buffer size will be used
  self.stdout = io.open(c2pread, 'rb', bufsize)
INFO:google.cloud.aiplatform.docker_utils.local_util:Using default tag: latest

INFO:google.cloud.aiplatform.docker_utils.local_util:The push refers to repository [us-central1-docker.pkg.dev/ridwan-faturrahman/random-forest-classifier-repo/random-forest-classifier-app]

INFO:google.cloud.aiplatform.docker_utils.local_util:92a2b4d6ec75: Waiting

INFO:google.cloud.aiplatform.docker_utils.local_util:866771c43bf5: Waiting

INFO:google.cloud.aiplatform.docker_utils.local_util:ed881fbf1b07: Waiting

INFO:google.cloud.a

### For Checking

In [37]:
models = aiplatform.Model.list()

for model in models:
    print("Name:", model.display_name)
    print("Resource:", model.resource_name)
    print("----")

In [38]:
endpoints = aiplatform.Endpoint.list()

for ep in endpoints:
    print("Name:", ep.display_name)
    print("Resource:", ep.resource_name)
    print("----")

### Upload Model and Create Endpoint

In [25]:
DOCKER_IMAGE_NAME

'us-central1-docker.pkg.dev/ridwan-faturrahman/random-forest-classifier-repo/random-forest-classifier-app'

In [26]:
model = aiplatform.Model.upload(
    display_name="random-forest-classifier-model",
    artifact_uri=RANDOM_FOREST_CLASSIFIER_ARTIFACT_DIR,
    serving_container_image_uri=DOCKER_IMAGE_NAME
)

Creating Model
Create Model backing LRO: projects/344969539300/locations/us-central1/models/8820070072274911232/operations/2011233962382327808
Model created. Resource name: projects/344969539300/locations/us-central1/models/8820070072274911232@1
To use this Model in another session:
model = aiplatform.Model('projects/344969539300/locations/us-central1/models/8820070072274911232@1')


In [27]:
endpoint = aiplatform.Endpoint.create(
    display_name="random-forest-classifier-endpoint"
)

print("Endpoint created:")
print(endpoint.resource_name)

Creating Endpoint
Create Endpoint backing LRO: projects/344969539300/locations/us-central1/endpoints/5382998373114576896/operations/7147448599910023168
Endpoint created. Resource name: projects/344969539300/locations/us-central1/endpoints/5382998373114576896
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/344969539300/locations/us-central1/endpoints/5382998373114576896')
Endpoint created:
projects/344969539300/locations/us-central1/endpoints/5382998373114576896


### Deploy Endpoint

In [28]:
model = aiplatform.Model(
    "projects/344969539300/locations/us-central1/models/8820070072274911232"
)

In [31]:
model.deploy(
    endpoint=endpoint,
    deployed_model_display_name="random-forest-classifier-deployed",
    machine_type="e2-standard-2",
    min_replica_count=1,
    max_replica_count=1,
)

Deploying model to Endpoint : projects/344969539300/locations/us-central1/endpoints/5382998373114576896
Deploy Endpoint model backing LRO: projects/344969539300/locations/us-central1/endpoints/5382998373114576896/operations/2887747039859310592
Endpoint model deployed. Resource name: projects/344969539300/locations/us-central1/endpoints/5382998373114576896


resource name: projects/344969539300/locations/us-central1/endpoints/5382998373114576896

### Inference

In [32]:
endpoint = aiplatform.Endpoint(
    "projects/344969539300/locations/us-central1/endpoints/5382998373114576896"
)

In [33]:
# instances = [
#     [1.0, 2.0, 3.0, 4.0, 1.0, 2.0, 3.0, 4.0, 9.0, 9.0],
#     [1.0, 2.0, 3.0, 4.0, 1.0, 2.0, 3.0, 4.0, 9.0, 11.0],
# ]
instances = [
    [6.7, 3.1, 4.7, 1.5],
    [4.6, 3.1, 1.5, 0.2]
]

response = endpoint.predict(instances=instances)

In [34]:
response

Prediction(predictions=['versicolor', 'setosa'], deployed_model_id='7259715188146831360', metadata=None, model_version_id='1', model_resource_name='projects/344969539300/locations/us-central1/models/8820070072274911232', explanations=None)

In [35]:
endpoint.predict(instances=[[6.7, 3.1, 4.7, 1.5], [4.6, 3.1, 1.5, 0.2]])

Prediction(predictions=['versicolor', 'setosa'], deployed_model_id='7259715188146831360', metadata=None, model_version_id='1', model_resource_name='projects/344969539300/locations/us-central1/models/8820070072274911232', explanations=None)

### Clean

In [36]:
# Clean
endpoint = aiplatform.Endpoint(
    "projects/344969539300/locations/us-central1/endpoints/5382998373114576896"
)

endpoint.delete(force=True)

model = aiplatform.Model(
    "projects/344969539300/locations/us-central1/models/8820070072274911232"
)

model.delete()

Undeploying Endpoint model: projects/344969539300/locations/us-central1/endpoints/5382998373114576896
Undeploy Endpoint model backing LRO: projects/344969539300/locations/us-central1/endpoints/5382998373114576896/operations/2028263198473322496
Endpoint model undeployed. Resource name: projects/344969539300/locations/us-central1/endpoints/5382998373114576896
Deleting Endpoint : projects/344969539300/locations/us-central1/endpoints/5382998373114576896
Endpoint deleted. . Resource name: projects/344969539300/locations/us-central1/endpoints/5382998373114576896
Deleting Endpoint resource: projects/344969539300/locations/us-central1/endpoints/5382998373114576896
Delete Endpoint backing LRO: projects/344969539300/locations/us-central1/operations/2974124673337393152
Endpoint resource projects/344969539300/locations/us-central1/endpoints/5382998373114576896 deleted.
Deleting Model : projects/344969539300/locations/us-central1/models/8820070072274911232
Model deleted. . Resource name: projects/3